<img src = "images.png">

Adjacency Matrix

In [3]:
import numpy as np

A = np.array([[0,1,0,0,1,0] , 
              [1,0,1,1,1,0] , 
              [0,1,0,1,0,1] ,
              [0,1,1,0,1,1] , 
              [1,1,0,1,0,1] ,
              [0,0,1,1,1,0]])

degree

In [4]:
du = A.sum(axis=1)

transition matrix

In [5]:
P = A/du[: , None]

random walk generator

In [6]:
def generate_walk(walk_length , num_nodes , P , start_node):
    walk = [start_node]
    curr = start_node
    for i in range(walk_length-1):
        next_node = np.random.choice(num_nodes , p=P[curr])
        walk.append(next_node)
        curr = next_node

    return walk


In [7]:
walk = generate_walk(10 , 6 ,P , 3)
print(walk)

[3, 2, 1, 0, 4, 5, 3, 2, 3, 4]


generate multiple walks

In [8]:
def generate_walks(P , num_walks , walk_length) :
    walks = []
    num_nodes = len(P)
    for i in range(num_walks):
        for start_node in range(num_nodes):
            walk = generate_walk(walk_length , num_nodes , P , start_node)
            walks.append(walk)
    return walks
    



context_pair generation

In [9]:
def context_pair(walk , window_size):
    pair = []
    for i in range(len(walk)):
        left = max(0 , i-window_size)
        right = min(len(walk) , i+window_size+1)

        for j in range(left , right) :
            if i==j:
                continue
            else:
                pair.append((walk[i], walk[j]))

    return pair


pair_list = context_pair(walk , 2 )
print(pair_list)


[(3, 2), (3, 1), (2, 3), (2, 1), (2, 0), (1, 3), (1, 2), (1, 0), (1, 4), (0, 2), (0, 1), (0, 4), (0, 5), (4, 1), (4, 0), (4, 5), (4, 3), (5, 0), (5, 4), (5, 3), (5, 2), (3, 4), (3, 5), (3, 2), (3, 3), (2, 5), (2, 3), (2, 3), (2, 4), (3, 3), (3, 2), (3, 4), (4, 2), (4, 3)]


generate all pairs


In [10]:
def generate_all_pairs(walks , window_size) :
    pairs = []
    for walk in walks:
        pair = context_pair(walk , window_size)
        pairs.extend(pair)
    return pairs


negative sample

In [11]:
def negative_sampling(num_nodes, positive) :
    negative = np.random.randint(0 , num_nodes)
    while positive == negative:
        negative = np.random.randint(0 , num_nodes)

    return negative

create training data

In [12]:
import torch 
import torch.nn as nn

def create_training_data(pairs , num_nodes):
    centers = []
    positives =[]
    negatives = []
    for center , positive in pairs :
        centers.append(center)
        positives.append(positive)
        negative = negative_sampling(num_nodes ,positive)
        negatives.append(negative)
    return (torch.tensor(centers) , torch.tensor(positives) , torch.tensor(negatives))



    

SKIPGRAM

In [13]:
class SkipGram(nn.Module):
    def __init__ (self , num_nodes , embedding_dim):
        super().__init__()

        self.center_embeddings = nn.Embedding(num_nodes , embedding_dim)
        self.context_embeddings = nn.Embedding(num_nodes , embedding_dim)

    def forward (self , centers , positives , negatives):
        v_center = self.center_embeddings(centers)
        v_positive = self.context_embeddings(positives)
        v_negative = self.context_embeddings(negatives)

        #score

        positive_score = (v_center*v_positive).sum(dim=1)
        negative_score = (v_center*v_negative).sum(dim=1)

        #loss
        positive_loss = -torch.log(torch.sigmoid(positive_score))
        negative_loss = -torch.log(torch.sigmoid(-negative_score))
        loss = (positive_loss+negative_loss).mean()
        return loss



In [14]:
model = SkipGram( num_nodes=6,
    embedding_dim=4)
optimizer = torch.optim.Adam(model.parameters() , lr = 0.1)

training loop

In [15]:
batch_size = 32
epochs = 20 

In [16]:
walks = generate_walks(P , 10 , 20)
pairs = generate_all_pairs(walks , 2)
centers , positives , negatives = create_training_data(pairs , 6)

In [17]:
import torch
for epoch in range(epochs) :
    total_loss = 0
    num_batches = 0
    permutation = torch.randperm(len(centers))
    for i in range(0 , len(centers) , batch_size) :
        indices = permutation[i:i+batch_size]
        batch_centers = centers[indices]
        batch_positives = positives[indices]
        batch_negatives = negatives[indices]

        optimizer.zero_grad()

        loss = model(batch_centers,batch_positives,batch_negatives)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        num_batches+=1
    print(f"Epoch : {epoch}")
    print(f"Avg loss : {total_loss/num_batches}")

Epoch : 0
Avg loss : 1.3879575772251156
Epoch : 1
Avg loss : 1.346911502399033
Epoch : 2
Avg loss : 1.348412057478651
Epoch : 3
Avg loss : 1.3469008944875045
Epoch : 4
Avg loss : 1.3547012291366247
Epoch : 5
Avg loss : 1.3559460588496366
Epoch : 6
Avg loss : 1.3586677518679942
Epoch : 7
Avg loss : 1.3478859096979923
Epoch : 8
Avg loss : 1.35308356936887
Epoch : 9
Avg loss : 1.3580614302655776
Epoch : 10
Avg loss : 1.3526691901598045
Epoch : 11
Avg loss : 1.3542870188788545
Epoch : 12
Avg loss : 1.3497551122157694
Epoch : 13
Avg loss : 1.3606042861938477
Epoch : 14
Avg loss : 1.3543734344647085
Epoch : 15
Avg loss : 1.3540606232855816
Epoch : 16
Avg loss : 1.352208307321123
Epoch : 17
Avg loss : 1.3532611551902276
Epoch : 18
Avg loss : 1.3619667831942333
Epoch : 19
Avg loss : 1.3549762166661323


In [18]:
embeddings = (
    model.center_embeddings.weight
    .detach()
    .cpu()
    .numpy()
)

print("Shape:", embeddings.shape)

for node in range(6):
    print(
        f"Node {node}:",
        embeddings[node]
    )

Shape: (6, 4)
Node 0: [ 0.7142929  -0.09597192  0.02069236 -1.3930489 ]
Node 1: [-0.63313544  0.27014247  0.2976726  -0.08943323]
Node 2: [-0.05130063 -1.1806085  -0.40416455  0.46690133]
Node 3: [ 0.47773683 -0.47802657  0.5911839  -0.04772872]
Node 4: [ 0.01495616 -0.46710354 -0.13568236 -0.6453074 ]
Node 5: [-0.62153465 -0.84803     0.2965969  -0.24612303]


In [19]:
!pip install -U scikit-learn




[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [20]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

np.set_printoptions(
    precision=3,
    suppress=True
)

print(similarity)

[[ 1.    -0.293 -0.278  0.33   0.751 -0.008]
 [-0.293  1.    -0.445 -0.37  -0.194  0.325]
 [-0.278 -0.445  1.     0.232  0.282  0.535]
 [ 0.33  -0.37   0.232  1.     0.249  0.294]
 [ 0.751 -0.194  0.282  0.249  1.     0.558]
 [-0.008  0.325  0.535  0.294  0.558  1.   ]]


In [21]:
for i in range(6):

    neighbors = np.argsort(
        -similarity[i]
    )

    print(
        f"Node {i}:",
        neighbors
    )

Node 0: [0 4 3 5 2 1]
Node 1: [1 5 4 0 3 2]
Node 2: [2 5 4 3 0 1]
Node 3: [3 0 5 4 2 1]
Node 4: [4 0 5 2 3 1]
Node 5: [5 4 2 1 3 0]


In [23]:
for k in [1, 2, 3]:

    precisions = []

    for node in range(6):

        # Exclude the node itself
        ranked = np.argsort(
            -similarity[node]
        )

        ranked = ranked[
            ranked != node
        ]

        top_k = ranked[:k]

        actual_neighbors = np.where(
            A[node] == 1
        )[0]

        precision = np.mean(
            np.isin(
                top_k,
                actual_neighbors
            )
        )

        precisions.append(precision)

    print(
        f"Precision@{k}: "
        f"{np.mean(precisions):.3f}"
    )

Precision@1: 0.667
Precision@2: 0.667
Precision@3: 0.611
